# Introduction to Pandas

<center><img src="https://geo-python-site.readthedocs.io/en/latest/_images/pandas_logo.png" height="300"/></center>

**Pandas** is one of the most powerful Python libraries for working with data. It gives you tools to load, clean, explore, and transform data — all in just a few lines of code. It was created by [Wes McKinney](https://en.wikipedia.org/wiki/Wes_McKinney), and its name comes from the term "**Pan**el **Da**ta", an econometrics term for datasets that track the same individuals over multiple time periods.

The core data structure in Pandas is the **DataFrame** — think of it like a spreadsheet or a database table, but with the full power of Python behind it. Almost everything we do as data scientists starts with getting our data into a DataFrame.

This notebook introduces you to DataFrames from the ground up: how to create them, how to load data into them, and how to explore and manipulate that data using Pandas' built-in tools.

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Explain what Pandas is and why it matters for data science

2. Create a Pandas **DataFrame** from a list of dictionaries or a list of lists

3. Load data from a CSV file into a DataFrame using `pd.read_csv()`

4. Explore a DataFrame using `.head()`, `.tail()`, `.info()`, `.describe()`, `.shape`, and `.columns`

5. Access specific rows and columns using bracket notation, `.loc[]`, and `.iloc[]`

6. Filter rows using **boolean masking** and the `.query()` method

7. Use **groupby**, **sorting**, and **column operations** to summarise and transform data

8. Handle missing values with `.fillna()` and `.dropna()`

---

## Pandas Import

Before we can use Pandas, we need to **import** it. This is a one-line step that gives our script access to everything in the library.

In [ ]:
# Standard import — always use 'pd' as the alias
import pandas as pd

The line `import pandas as pd` loads the entire Pandas library and makes it accessible via the shorthand `pd`.

Using `pd` as the alias is **standard practice** across the Python data science community. Sticking to conventions like this makes your code easier for others to read and understand.

> **Tip:** Comments (lines starting with `#`) are notes for humans — Python ignores them. Use them freely to explain your thinking.

---

## Getting a DataFrame Object

There are two ways to get a DataFrame to work with:

1. **From data already in your Python program** — using the `pd.DataFrame()` constructor

2. **From an external file** — using `pd.read_csv()`, `pd.read_excel()`, and similar functions

Let's start with building DataFrames from scratch. See the [DataFrame docs](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html) for full reference.

### Method 1: From a List of Dictionaries

Pass a **list of dictionaries** to `pd.DataFrame()`. Each dictionary becomes **one row**. The dictionary keys become the **column names**.

In [ ]:
# Create a DataFrame from a list of dictionaries
# Each dictionary becomes one row; the keys become column names
data_lst = [{'a': 1, 'b': 2, 'c': 3}, {'a': 4, 'b': 5, 'c': 6, 'd': 7}]
df_example = pd.DataFrame(data_lst)
df_example

Notice that the first dictionary has no `'d'` key. Pandas fills the missing value with **`NaN`** (Not a Number) — the standard way Pandas represents missing data.

Now let's go further — what if *every* dictionary is missing some keys? Pandas still builds the full table, placing `NaN` wherever a value is absent.

**What do you expect the DataFrame below to look like?**

In [ ]:
# Each dict has a different key — Pandas creates a column for every unique key it finds
data_lst = [{'a': 1}, {'b': 5}, {'c': 4}]
df_sparse = pd.DataFrame(data_lst)
df_sparse

### Method 2: From a List of Lists

Pass a `data` argument (a list of lists) and a `columns` argument (a list of column names). Each inner list becomes **one row**.

The number of values in each list must match the number of column names you provide.

In [ ]:
# Two rows of 3 values each, matched to 3 column names
data_vals = [[1, 2, 3], [4, 5, 6]]
data_cols = ['a', 'b', 'c']
df_example2 = pd.DataFrame(data=data_vals, columns=data_cols)
df_example2

Unlike the list-of-dictionaries method, this approach is stricter: if any row has **more values** than the number of columns you declared, Pandas will raise a `ValueError`. The cell below triggers that error intentionally — run it to see what it looks like.

In [ ]:
# This raises a ValueError: the second row has 3 values but only 2 columns are declared
data_vals = [[1, 2], [4, 5, 6]]
data_cols = ['a', 'b']
df_err = pd.DataFrame(data=data_vals, columns=data_cols)  # <-- will raise ValueError

If a row has **fewer** values than the number of columns, Pandas is more forgiving — the missing trailing values are filled with `NaN`.

In [ ]:
# First row has 2 values, but 3 columns are declared — the missing 'c' becomes NaN
data_vals = [[1, 2], [4, 5, 6]]
data_cols = ['a', 'b', 'c']
df_example3 = pd.DataFrame(data=data_vals, columns=data_cols)
df_example3

### Method 3: Reading External Data

In practice, data lives in files — CSVs, Excel sheets, JSON, SQL databases. Pandas has a `read_*` function for each format. The most common is `pd.read_csv()`.

```python
# Basic usage — assumes column names are in the first row
df = pd.read_csv('my_data.csv')

# If there are no column headers in the file:
df = pd.read_csv('my_data.csv', header=None)

# Assign your own column names as you load:
df = pd.read_csv('my_data.csv', header=None, names=['col1', 'col2', 'col3'])

# If the file uses a separator other than a comma (e.g. semicolons):
df = pd.read_csv('my_data.csv', sep=';')
```

For the full list of options, see the [`pd.read_csv()` documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

---

## Looking at Our Data

Our dataset is `abalone.csv` — Let's load it and start exploring.

In [ ]:
# Load the abalone dataset from the shared data folder
# The path goes one level up (../) because this notebook is inside the 1_Pandas/ folder
df = pd.read_csv('../data/abalone.csv')

A DataFrame has **attributes** and **methods** that help you understand what's inside it before you start working with it. The key ones are:

- **Attributes** (no parentheses): `.shape`, `.columns`

- **Methods** (with parentheses): `.head()`, `.tail()`, `.info()`, `.describe()`

Let's work through each one.

In [ ]:
# Show the first 5 rows (default)
df.head()

With this data we are taking a trip under the sea! Abalones are large sea snails found along coastlines around the world.

<center><img src="https://upload.wikimedia.org/wikipedia/commons/b/bc/Abalone_at_California_Academy_of_Sciences.JPG" height="400"/></center>

The dataset comes from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Abalone). The original research goal was to **predict the age of an abalone** from physical measurements — a much faster (and kinder) alternative to the traditional method of cutting the shell and counting growth rings under a microscope.

**Column descriptions:**

| Column | Type | Unit | Description |
|:---|:---|:---|:---|
| `sex` | nominal | — | M (male), F (female), I (infant) |
| `length` | continuous | mm | Longest shell measurement |
| `diameter` | continuous | mm | Perpendicular to length |
| `height` | continuous | mm | With meat in shell |
| `weight_whole` | continuous | grams | Whole abalone |
| `weight_shucked` | continuous | grams | Weight of meat only |
| `viscera` | continuous | grams | Gut weight after bleeding |
| `shell` | continuous | grams | Shell weight after drying |
| `n_rings` | integer | — | Number of rings (+1.5 = age in years) |

In [ ]:
# Pass a number to see more rows — useful when you want a bigger preview
df.head(10)

`.tail()` shows the **last** rows of the DataFrame — useful for checking that your data loaded completely.

In [ ]:
# Show the last 5 rows
df.tail()

`.shape` returns a tuple `(rows, columns)`. `.columns` returns all column names as an **Index** object — similar to a list.

In [ ]:
# (rows, columns) — no parentheses, it's an attribute not a method
df.shape

In [ ]:
# Returns an Index of all column names
df.columns

We have **4,177 rows** and **9 columns**.

`.info()` goes deeper — it shows the **data type** of each column and how many **non-null** values are present. This helps you quickly spot missing data and verify that columns loaded with the right types.

In [ ]:
# Summary of column types and non-null counts
df.info()

All 4,177 rows have values in every column — no missing data here.

`.describe()` gives you a **statistical summary** of all numeric columns: count, mean, standard deviation, min, max, and the three quartiles. It's a quick way to get a feel for the spread of your data.

In [ ]:
# Statistical summary of all numeric columns
df.describe()

Notice that `sex` is missing from `.describe()` — it only summarises **numeric** columns. The `count` row tells you how many non-null values each column has. Since all counts equal 4,177, we have no missing values in this dataset.

---

## Accessing Data in a DataFrame

Now that we can load and inspect data, let's learn how to **select** specific parts of it.

There are several ways to grab data from a DataFrame, and the right approach depends on what you want:

- **A single column** → bracket notation or dot notation

- **Multiple columns** → pass a list inside brackets

- **A slice of rows** → slice notation

- **Specific rows AND columns** → `.loc[]` (label-based) or `.iloc[]` (integer-based)

### Selecting Columns

Use the **column name** in brackets to get an entire column. This returns a **Series** — a one-dimensional array with an index attached (more on this later).

In [ ]:
# Bracket notation — always works
df['sex']

In [ ]:
# Dot notation — shorter, but has limitations (see note below)
df.sex

> **Note:** Dot notation (`df.sex`) fails if the column name contains **spaces** or clashes with an existing DataFrame method. Bracket notation (`df['sex']`) always works and is the safer choice.

If you ever load data with spaces in column names, here's a quick fix:

In [ ]:
# Replace spaces with underscores in all column names
df2 = df.copy()
df2.columns = [col.replace(' ', '_') for col in df2.columns]
df2.columns

To select **multiple columns** at once, pass a **list** of column names inside the brackets. This returns a DataFrame (not a Series).

In [ ]:
# Select two columns at once — notice the double brackets [[ ]]
df[['sex', 'length']]

### Selecting Rows

You can **slice** rows using Python's standard slice notation — the same way you would slice a list.

In [ ]:
# Rows 0, 1, 2 — up to but not including index 3
df[:3]

# A single integer raises a KeyError — Pandas looks for a column named 0, not row 0
# Use a slice or .loc/.iloc to get specific rows

# df[0]  # <-- uncomment and run to see: KeyError: 0

# You also can't select rows AND columns at the same time with plain brackets:

# df[:1, 'sex']  # <-- uncomment and run to see: TypeError

### Selecting Rows & Columns: `.loc[]` and `.iloc[]`

When you need to select **specific rows and specific columns at the same time**, use one of these two indexers:

| Indexer | Takes | Example |
|:---|:---|:---|
| `.loc[]` | **Labels** (column names, index labels) | `df.loc[0, 'sex']` |
| `.iloc[]` | **Integer positions** (0, 1, 2...) | `df.iloc[0, 0]` |

Think of it this way: `.loc` = **l**abels, `.iloc` = **i**ntegers.

In [ ]:
# Single value: row with index label 0, column named 'sex'
df.loc[0, 'sex']

In [ ]:
# Range of rows (inclusive on both ends with .loc), single column
df.loc[0:10, 'sex']

In [ ]:
# Range of rows, multiple columns passed as a list
df.loc[10:15, ['sex', 'length']]

In [ ]:
# .loc[] raises a KeyError when you pass integer column positions — it only accepts labels.
# Uncomment any of these to see the error:
# df.loc[0, 0]           # KeyError: 0 is not a column label
# df.loc[0:10, 0]        # KeyError
# df.loc[10:15, [0, 4]]  # KeyError

In [ ]:
# .iloc[] works with integer positions — it does NOT accept column labels
print(df.iloc[0, 0])        # single value: row 0, column 0
print(df.iloc[0:5, 0])      # first 5 rows, first column
df.iloc[10:15, [0, 4]]      # rows 10-14, columns 0 and 4 by position

In [ ]:
# .iloc[] raises a ValueError when you pass a label — it only accepts integers.
# Uncomment to see the error:
# df.iloc[0, 'sex']  # ValueError: cannot label-index with a string using iloc

### Filtering with Boolean Masks and `.query()`

So far we've selected data by **position** or **label**. But often we want to select rows based on a **condition** — for example, "show me all abalones with a length under 0.4mm".

**Step 1:** A condition on a column returns a **boolean mask** — a Series of `True`/`False` values, one per row.

In [ ]:
# This returns True/False for every row — not the data itself yet
df['length'] <= 0.4

**Step 2:** Use the mask **inside brackets** to filter the DataFrame — only rows where the mask is `True` are returned.

In [ ]:
# Only rows where length is 0.4 or less
df[df['length'] <= 0.4]

You can combine multiple conditions using `&` (AND) and `|` (OR). Each condition **must** be wrapped in its own parentheses.

In [ ]:
# Female abalones with length 0.4 or less
df[(df['length'] <= 0.4) & (df['sex'] == 'F')]

In [ ]:
# A more complex filter — multiple conditions combined with &
df[(df['length'] <= 0.4) & (df['length'] > 0.35) & (df['weight_whole'] > 0.25) & (df['weight_whole'] < 0.3)]

The `.query()` method lets you write the same filter as a **readable string**. It uses `and`/`or` instead of `&`/`|`, and drops the heavy bracket nesting. For complex filters, `.query()` is much easier to read.

In [ ]:
# Same filter as above, written as a readable string
df.query('length <= 0.4 and length > 0.35 and weight_whole > 0.25 and weight_whole < 0.3')

For a complete list of DataFrame attributes and methods, see the [Pandas DataFrame documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html). For a deep-dive with real-world examples, the book [Python for Data Analysis](http://shop.oreilly.com/product/0636920023784.do) (by Wes McKinney, the creator of Pandas) is an excellent resource.

---

## Groupby, Sorting, and Column Operations

### Groupby

**Groupby** is one of the most powerful tools in Pandas. It follows a **split → apply → combine** pattern:

1. **Split** the DataFrame into groups based on a column#

2. **Apply** an aggregation function (mean, count, max, etc.) to each group

3. **Combine** the results back into a single DataFrame

This is equivalent to a `GROUP BY` clause in SQL or a pivot table in Excel.

In [ ]:
# Check how many unique categories exist in the 'sex' column
df['sex'].unique()

In [ ]:
# Count the number of distinct values — useful to know before grouping
df['sex'].nunique()

In [ ]:
# Calling groupby() alone returns a GroupBy object — it doesn't compute anything yet.
# You need to chain it with an aggregation method to get results.
df.groupby('sex')

Store the groupby object in a variable when you plan to run multiple aggregations on it — this is more efficient than calling `.groupby()` repeatedly.

In [ ]:
# Store the groupby object for reuse
groupby_sex = df.groupby('sex')

# --- Mean of all numeric columns, grouped by sex ---
# Pandas 3.0 note: pass numeric_only=True to avoid warnings on non-numeric columns
groupby_sex.mean(numeric_only=True)

In [ ]:
# Maximum value in each group
groupby_sex.max(numeric_only=True)

In [ ]:
# Count of non-null values per column in each group
groupby_sex.count()

The groupby column (`sex`) becomes the **index** of the result. To extract just one column from the grouped result — for example, the count per sex group — chain it with bracket notation.

In [ ]:
# How many abalones of each sex are in the dataset?
df.groupby('sex').count()['length']

You can group by **multiple columns** by passing a list. Pandas groups by the first column, then by the second within each of those groups — creating a **MultiIndex** result.

In [ ]:
# Group by sex, then by ring count — how many abalones in each sex/ring combination?
df.groupby(['sex', 'n_rings']).count()['weight_whole']

See all available aggregation options in the [Groupby documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/groupby.html).

### Sorting

`.sort_values()` sorts the DataFrame by one or more columns. By default it sorts **ascending** (smallest first).

In [ ]:
# Sort by length ascending (default), show the 15 smallest abalones
df.sort_values('length').head(15)

# To reverse the order: df.sort_values('length', ascending=False)

To sort by **multiple columns**, pass a list. Pandas sorts by the first column first, then uses the second to break ties.

In [ ]:
# Sort by length descending, then diameter descending (to break ties)
df.sort_values(['length', 'diameter'], ascending=False).head(15)

### Creating and Dropping Columns

You can **create** a new column using:
1. **Direct assignment**: `df['new_col'] = expression`
2. **`df.eval()`**: write the expression as a string — useful for readable, concise column creation

You can **drop** a column with `df.drop()`. Remember to specify `axis=1` (columns) vs `axis=0` (rows).

![Axis diagram](https://i.stack.imgur.com/DL0iQ.jpg)

In [ ]:
# Rename the 'n_rings' column to 'nr_rings' for clarity
df.rename(columns={'n_rings': 'nr_rings'}, inplace=True)
df.head()

In [ ]:
# Create a new 'age' column — per the dataset: age (years) = nr_rings + 1.5
# Pandas 3.0 note: eval() with inplace=True is deprecated — assign the result directly
df = df.eval('age = nr_rings + 1.5')
df.columns

In [ ]:
# Create another derived column: weight relative to diameter
# Pandas 3.0 note: eval() with inplace=True is deprecated — assign the result directly
df = df.eval('weight_per_diam = weight_whole / diameter')
df.columns

In [ ]:
# Verify the new columns were created correctly
df.head()

Now that we have `age`, the `nr_rings` column is redundant — both encode the same information. Let's drop it.

By default, `df.drop()` returns a **new** DataFrame without the column, leaving the original unchanged. Pass `inplace=True` to modify `df` directly.

In [ ]:
# Without inplace=True, the result is returned but df is NOT modified
df.drop('nr_rings', axis=1).columns

In [ ]:
# nr_rings is still in df — the drop was not applied
df.columns

In [ ]:
# Now drop it for real — inplace=True modifies df directly
df.drop('nr_rings', axis=1, inplace=True)

In [ ]:
# Confirm nr_rings is gone
df.columns

### Dealing with Null Values

Real-world datasets almost always have **missing values** (represented as `NaN` in Pandas). You have two main options:

- **`.fillna(value)`** — replace `NaN` with a specific value (e.g. `0`, `-1`, or the column mean)

- **`.dropna()`** — remove any row that contains a `NaN`

The right choice depends on your data and your goal. Read more in the [Missing Data guide](https://pandas.pydata.org/pandas-docs/stable/user_guide/missing_data.html).

In [ ]:
# Our abalone dataset has no missing values, so these lines have no visible effect here.
# They demonstrate the pattern you'll use on datasets that do have NaNs.

# --- Replace NaN with a placeholder value ---
df.fillna(-1, inplace=True)

# --- Drop any row that contains a NaN ---
df.dropna(inplace=True)

---

## A Quick Aside: Pandas Series

You may have noticed that selecting a single column returns something slightly different from a DataFrame — it has a column of values with an index attached. This is called a **`Series`**.

A **Series** is a one-dimensional labelled array. Think of it as a single column from a DataFrame. Most DataFrame methods also work on Series.

Here are some examples from cells we've already run:

In [ ]:
# A boolean condition on a column returns a Series of True/False values
df['length'] <= 0.3

In [ ]:
# Confirm it's a Series object
type(df['length'] <= 0.3)

In [ ]:
# Selecting one column from a groupby result also returns a Series
df.groupby('sex').count()['age']

---

## Key Takeaways

- **Pandas** is the standard library for working with tabular data in Python

- The **DataFrame** is your main data structure — think of it as a programmable spreadsheet

- Always **explore before you transform**: use `.head()`, `.info()`, `.describe()`, `.shape`

- Use **bracket notation** or `.loc[]` / `.iloc[]` to access rows and columns

- **Boolean masking** and `.query()` let you filter rows by condition

- **Groupby** aggregates data across categories (split → apply → combine)

- Handle missing data with `.fillna()` or `.dropna()` — never ignore `NaN`s

---

## Check Your Understanding

Answer these questions mentally or in a scratch cell before moving on:

1. What is the difference between `.loc[]` and `.iloc[]`? Give one example of each.

2. You run `df['age'].mean()` and get `NaN`. What is the most likely cause?

3. What does `.describe()` NOT show, and why?

4. You want to find all rows where `weight_whole > 1.0` AND `sex == 'M'`. Write the boolean mask filter.

5. What is the difference between `df.drop('col', axis=1)` and `df.drop('col', axis=1, inplace=True)`?

### **Ready to practice?** Open [Notebook_2](02_Pandas_Practice_Functionalities.ipynb) to apply what you've learned.

---

## Further Study

There are many more methods and features in Pandas. Here are some resources to go deeper:

- [Helpful Python Code Snippets for Data Exploration in Pandas](https://medium.com/@msalmon00/helpful-python-code-snippets-for-data-exploration-in-pandas-b7c5aed5ecb9)

- [Manipulating tabular data with Pandas](https://neuroimaging-data-science.org/content/004-scipy/002-pandas.html)

- [Book — Python for Data Analysis (Wes McKinney)](http://shop.oreilly.com/product/0636920023784.do)

- [Pandas official documentation](https://pandas.pydata.org/pandas-docs/stable/)